# Phase 1 — Data preparation (Colab)

ADSP 32023 CV Final Project — Severstal Steel Defect Detection

Runs the whole phase-1 chain **inside Colab**, because Colab is the delivery environment:
package versions, filesystem layout and Drive mounting all differ from a local machine, so
anything verified only locally is not actually verified.

**What this notebook does**
1. mount Drive and clone the repo
2. download the competition data (~1.7 GB) into the shared Drive folder
3. **confirm the real image dimensions** from an actual file
4. round-trip the RLE codec against real `train.csv` rows
5. build `index.csv` including the defect-free images
6. generate and freeze the stratified train/val split

**Prerequisites** (one-time, done outside this notebook)
- joined the Severstal competition on kaggle.com (requires phone verification)
- a Kaggle API token available as `KAGGLE_API_TOKEN`

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cv final project'   # the shared team folder
DATA_DIR   = f'{DRIVE_ROOT}/data'                        # 1.7 GB lives here, shared by all 3
REPO_DIR   = '/content/cv-steel-defect'                  # code lives in git, not Drive

import os
os.makedirs(DATA_DIR, exist_ok=True)
print('Drive folder :', DRIVE_ROOT, '| exists:', os.path.isdir(DRIVE_ROOT))
print('Data dir     :', DATA_DIR)

In [ ]:
# Code comes from git and is READ-ONLY here: edit locally, commit, pull. Editing src/ in a
# Colab cell would leave the change inside a throwaway runtime and invisible to teammates.
import os, subprocess
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/AZIO-126/cv-steel-defect.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
import sys
sys.path.insert(0, f'{REPO_DIR}/src')
print(subprocess.run(['git','log','--oneline','-1'], cwd=REPO_DIR,
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U kaggle kagglehub
!pip list 2>/dev/null | grep -Ei '^(kaggle|kagglehub|torch|pandas) ' 

## 2. Authenticate

Paste the token into the Colab **Secrets** panel (left sidebar, key icon) as `KAGGLE_API_TOKEN`
rather than hard-coding it in a cell — a token written into the notebook gets committed to git
and shared with the team.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    print('token loaded from Colab Secrets')
except Exception as e:
    print('Secrets unavailable (%s). Falling back to a prompt.' % type(e).__name__)
    import getpass
    os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('KAGGLE_API_TOKEN: ')

COMP = 'severstal-steel-defect-detection'
API  = f'https://www.kaggle.com/api/v1/competitions/data'
HDR  = {'Authorization': 'Bearer ' + os.environ['KAGGLE_API_TOKEN']}

In [ ]:
import requests
# A 200 here proves the token is valid but NOT that downloads will work: listing metadata
# does not require accepting the competition rules, downloading does. So test a real file.
r = requests.get(f'{API}/download/{COMP}/sample_submission.csv', headers=HDR, timeout=60)
print('sample_submission.csv ->', r.status_code, len(r.content), 'bytes')
assert r.status_code == 200 and not r.content.startswith(b'{"code":403'), (
    'Rules not accepted yet — join the competition on kaggle.com first. Body: %s'
    % r.content[:200])
print('rules gate: OK')

## 3. Download the competition data into Drive

~1.7 GB. It goes to Drive so it is downloaded **once** and read by all three members, and so a
runtime disconnect does not force a re-download.

In [ ]:
import os, requests, zipfile, time

ZIP_PATH = f'{DATA_DIR}/{COMP}.zip'

if os.path.exists(f'{DATA_DIR}/train.csv') and os.path.isdir(f'{DATA_DIR}/train_images'):
    print('data already present in Drive — skipping download')
else:
    if not os.path.exists(ZIP_PATH):
        print('downloading (this takes a few minutes) ...')
        t0 = time.time()
        with requests.get(f'{API}/download-all/{COMP}', headers=HDR,
                          stream=True, timeout=1800) as resp:
            resp.raise_for_status()
            total = 0
            with open(ZIP_PATH, 'wb') as f:
                for chunk in resp.iter_content(1 << 22):   # 4 MB chunks
                    f.write(chunk); total += len(chunk)
                    if total % (1 << 28) < (1 << 22):
                        print(f'  {total/1e9:.2f} GB', flush=True)
        print(f'downloaded {total/1e9:.2f} GB in {time.time()-t0:.0f}s')
    print('unzipping ...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print('done')

print(sorted(os.listdir(DATA_DIR))[:10])

## 4. Confirm the real image geometry

Not an assumption. The RLE codec is **column-major**, so passing a transposed shape produces
transposed masks with no error raised. `409600` total pixels is ambiguous on its own — it also
factors as 128x3200 and 640x640 — so read the dimensions off actual files.

In [ ]:
import os, random
from PIL import Image

train_dir = f'{DATA_DIR}/train_images'
files = sorted(os.listdir(train_dir))
sizes = {}
for fn in random.Random(0).sample(files, 10):
    sizes.setdefault(Image.open(f'{train_dir}/{fn}').size, 0)
    sizes[Image.open(f'{train_dir}/{fn}').size] += 1
print('distinct (W, H) over 10 random train images:', sizes)

(W, H), = sizes.keys()          # fails loudly if the dataset is not uniform
IMAGE_SHAPE = (H, W)            # what rle_decode expects
print(f'W x H = {W} x {H}  |  total px = {W*H}  |  rle shape (H, W) = {IMAGE_SHAPE}')
assert W * H == 409600, 'unexpected pixel count'
print('images in train_images:', len(files))

## 5. RLE round-trip against real annotations

In [ ]:
!cd {REPO_DIR} && python src/test_rle.py --csv "{DATA_DIR}/train.csv" 

## 6. Build index.csv — now including the defect-free images

`train.csv` holds one row per defect instance, so defect-free images are absent from it
entirely. They must come from the image directory, otherwise the class imbalance vanishes and
every later metric is computed on the wrong population.

In [ ]:
!cd {REPO_DIR} && python src/build_index.py \
    --train-csv "{DATA_DIR}/train.csv" \
    --images-dir "{DATA_DIR}/train_images" \
    --out "{DATA_DIR}/index.csv"
!cp "{DATA_DIR}/index.csv" {REPO_DIR}/data/index.csv

## 7. Freeze the split

One split for all four models across phases 3 and 4 — without it the champion-challenger
comparison compares models on different data and is void.

In [ ]:
!cd {REPO_DIR} && python src/split.py --index "{DATA_DIR}/index.csv" --outdir splits
!cd {REPO_DIR} && python src/split.py --index "{DATA_DIR}/index.csv" --verify

## 8. Sanity view — decoded masks on real images

The round-trip test proves the codec is self-consistent. This proves the masks land on actual
surface defects, which a transposed-but-self-consistent decode would not.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from rle import rle_decode

tr = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['EncodedPixels'])
sample = tr.sample(5, random_state=0)

fig, axes = plt.subplots(5, 1, figsize=(16, 10))
for ax, row in zip(axes, sample.itertuples()):
    img = np.array(Image.open(f'{train_dir}/{row.ImageId}').convert('RGB'))
    m = rle_decode(row.EncodedPixels, IMAGE_SHAPE)
    overlay = img.copy(); overlay[m == 1] = [255, 0, 0]
    ax.imshow(np.hstack([img, overlay])); ax.axis('off')
    ax.set_title(f'{row.ImageId}  class {row.ClassId}  area {int(m.sum())} px', fontsize=9)
plt.tight_layout()
os.makedirs(f'{DRIVE_ROOT}/outputs/figs', exist_ok=True)
plt.savefig(f'{DRIVE_ROOT}/outputs/figs/phase1_mask_overlay.png', dpi=110, bbox_inches='tight')
plt.show()

## 9. Phase 1 DONE checklist

- [ ] rules gate passed (a real file downloaded, not just metadata listed)
- [ ] image geometry read off actual files and uniform across samples
- [ ] RLE round-trip 9/9 including real `train.csv` rows
- [ ] `index.csv` includes defect-free images, so the true class balance is known
- [ ] split frozen, no train/val overlap, defect prevalence gap < 1 pp
- [ ] mask overlays visually land on real defects

Then commit `data/index.csv`, `splits/*.csv` and this executed notebook back to git.
Model weights stay in Drive and never enter the repo.